# Preprocesamiento

## 1. Carga del archivo

In [ ]:
import pyreadstat
import re
import unicodedata
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import prince
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

df, meta = pyreadstat.read_sav("/Users/tomas/github/eaui_subtel/data/sav/2026.sav")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

## 2. GSE derivado

In [ ]:
def _educ_g(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3:  return 'basica'
    if e <= 7:  return 'media'
    if e <= 9:  return 'tecnica'
    return 'universitaria'

_M = {
    (1,'basica'):'E',  (1,'media'):'E',  (1,'tecnica'):'D',  (1,'universitaria'):'D',
    (2,'basica'):'E',  (2,'media'):'D',  (2,'tecnica'):'D',  (2,'universitaria'):'C3',
    (3,'basica'):'D',  (3,'media'):'C3', (3,'tecnica'):'C3', (3,'universitaria'):'C2',
    (4,'basica'):'C3', (4,'media'):'C2', (4,'tecnica'):'C2', (4,'universitaria'):'C1',
    (5,'basica'):'C2', (5,'media'):'C1', (5,'tecnica'):'C1', (5,'universitaria'):'AB',
    (6,'basica'):'C1', (6,'media'):'AB', (6,'tecnica'):'AB', (6,'universitaria'):'AB',
}
_ORDEN_GSE = ['AB','C1','C2','C3','D','E']  # Invertido

_eg = df['A10'].apply(_educ_g)
df['gse'] = pd.Categorical(
    df['A11'].combine(_eg, lambda o, e: np.nan if pd.isna(o) or e is None else _M.get((int(o), e), np.nan)),
    categories=_ORDEN_GSE, ordered=True  # Aquí usa el orden invertido automáticamente
)
print('GSE:', df['gse'].value_counts().reindex(_ORDEN_GSE).to_dict())

## 3. Etiquetas limpias

In [ ]:
def limpiar_etiqueta(label):
    """Extrae la parte descriptiva útil de una etiqueta SPSS."""
    if not label: return label
    label = label.strip()
    # Patrón B/C: empieza con código de variable (P3_1 .-, Q1.3.-)
    if re.match(r'^[A-Z]\w+[\._]\w+\s*\.-?', label):
        if ':' in label:
            r = label.split(':')[-1].strip()
            if r: return r
        if '?' in label:
            r = label.split('?')[-1].strip().lstrip(':').strip()
            if r: return r
        r = re.sub(r'^[A-Z]\w+[\._]\w+[\s\._\-]+', '', label).strip()
        return r.lstrip('.-').strip()
    # Patrón A: etiqueta + [pregunta padre] (corchete abre, cierre o no)
    if '[' in label:
        r = label[:label.index('[')].strip()
        return re.sub(r'^\d+[\.-]+\s*', '', r).strip()
    # Patrón D: numeración inicial
    return re.sub(r'^\d+[\.-]+\s*', '', label).strip()


etiquetas_limpias = {
    col: limpiar_etiqueta(label)
    for col, label in zip(meta.column_names, meta.column_labels) if label
}
print(f"Etiquetas procesadas: {len(etiquetas_limpias)}")

## 4. Diccionario de variables originales

In [ ]:
diccionario = pd.DataFrame({'variable': meta.column_names, 'etiqueta': meta.column_labels})
diccionario


In [67]:
# Buscar variable por nombre o fragmento de etiqueta
busqueda = 'Q1'
diccionario[
    diccionario['variable'].str.contains(busqueda, case=False) |
    diccionario['etiqueta'].str.contains(busqueda, case=False, na=False)
]

,variable,etiqueta
177,Q1,Q1.- ¿Cuál es su parentesco con el Jefe /a de ...
178,Q1_1,Q1_1.- ¿Cuál es su edad?
179,Q1_2,Q1_2.- Sexo del entrevistado
180,Q1_3,Q1.3.- ¿Cuál fue el último curso y tipo de est...
181,Q1_3_1,Q1.3.- ¿Cuál fue el último curso y tipo de est...
182,Q1_4,Q1_4.- ¿Cuál es su profesión o trabajo o activ...
183,Q1_4_1,Q1.3.- ¿Cuál fue el último curso y tipo de est...
184,Q1_5_1,Aymara [Q1.5.- ¿Usted pertenece o es descendie...
185,Q1_5_2,Rapa-Nui [Q1.5.- ¿Usted pertenece o es descend...
186,Q1_5_3,Quechua [Q1.5.- ¿Usted pertenece o es descendi...


In [74]:
# Ver categorías codificadas de una variable
variable = 'A9'
labels = meta.variable_value_labels.get(variable, {})
if labels:
    for k, v in labels.items(): print(f'  {k} -> {v}')
else:
    print(f"'{variable}' no tiene etiquetas de valor.")

  1.0 -> Yo soy el jefe de Hogar
  2.0 -> Cónyuge o Pareja
  3.0 -> Hijo(a)
  4.0 -> Yerno o Nuera
  5.0 -> Nieto(a)
  6.0 -> Padre o Madre
  7.0 -> Suegro(a)
  8.0 -> Hermano(a)
  9.0 -> Cuñado(a)
  10.0 -> Otro familiar
  11.0 -> Otro no familiar
  12.0 -> Servicio Doméstico


## 5. Tratamiento NS/NR

Variables de montos usan `9999999` como código NS/NR.

In [ ]:
cols_nsnr = [
    'P11','Q7_4',
    'P17_1','P17_2','P17_3','P17_4','P17_5',
    'P19_1','P19_2','P19_3','P19_4',
    'Q40_1','Q40_2','Q40_3','Q40_4','Q40_5',
    'Q42','Q42_1'
]
for col in cols_nsnr:
    if col in df.columns: df[col] = df[col].replace(9999999, np.nan)
print('NS/NR reemplazados por NaN.')

## 6. Renombrado de variables

In [ ]:
nombres_cortos = {
    'REGISTRO':'id', 'FECHAFIN':'fecha_fin', 'COD_REGION':'region', 'COMUNA_DEF':'comuna', 'ZONA':'zona',
    'A9':'parentesco_jh', 'A10':'educ_jh', 'A11':'ocupacion_jh', 'A12_1':'ingreso_hogar',
    'Q1':'parentesco', 'Q1_1':'edad', 'Q1_2':'sexo', 'Q1_3':'educ', 'Q1_4':'ocupacion_encuestado', 'Q2':'actividad',
    'P1':'acceso_internet_hogar', 'P2':'n_smartphones_hogar', 'P2_1':'n_computadores_hogar',
    'P10':'tipo_acceso_fijo', 'P11':'pago_mensual_internet', 'P11_3':'velocidad_contratada',
    'P11_4':'calidad_acceso', 'P11_5':'cuota_mensual_gb', 'P12_2':'tipo_plan', 'P12_1':'plan_movil_tipo',
    'P14':'razon_no_acceso_principal', 'P15':'disposicion_contratar_fijo',
    'Q5':'uso_computador', 'Q7':'uso_smartphone', 'Q7_1':'smartphone_propio',
    'Q7_3':'plan_movil_tipo_ind', 'Q7_4':'pago_mensual_movil',
    'Q9':'ultimo_uso_internet', 'Q10':'frecuencia_internet', 'Q11':'tiempo_diario_internet',
    'Q13':'tipo_acceso_mas_usado', 'Q14':'uso_internet_hogar', 'Q15':'frecuencia_internet_hogar',
    'Q16':'tiempo_diario_hogar', 'Q17':'uso_internet_fuera_hogar', 'Q18':'frecuencia_fuera_hogar',
    'Q19':'tiempo_diario_fuera_hogar',
    'Q23':'internet_facilita_trabajo', 'Q25':'internet_mejora_vida', 'Q27':'ultima_compra_online',
    'Q31':'percepcion_proteccion', 'Q30_1':'reg_control_legal', 'Q30_2':'reg_control_familia', 'Q30_3':'reg_autocontrol',
    'FE_HOGAR':'fe_hogar', 'FE_PERSONAS':'fe_personas', 'POND_HOGAR':'pond_hogar', 'POND_PERSONAS':'pond_personas',
}

df = df.rename(columns={k: v for k, v in nombres_cortos.items() if k in df.columns})

# Recodificación de educ_jh y ocupacion_jh (aquí, con valores numéricos aún intactos)
_mapa_educ = {
    1:'Sin educación formal', 2:'Básica incompleta', 3:'Básica completa',
    4:'Media CH incompleta', 5:'Media TP incompleta', 6:'Media CH completa', 7:'Media TP completa',
    8:'Superior técnica incompleta', 9:'Superior técnica completa',
    10:'Superior universitaria incompleta', 11:'Superior universitaria completa'
}
_mapa_ocup = {
    1:'Trabajos ocasionales e informales', 2:'Oficio menor - obrero no calificado',
    3:'Obrero calificado - microempresario', 4:'Empleado medio - técnico - prof. independiente',
    5:'Ejecutivo medio - prof. universitario', 6:'Alto ejecutivo - empresario - directivo'
}
df['educ_jh']              = df['educ_jh'].map(_mapa_educ)
df['ocupacion_jh']         = df['ocupacion_jh'].map(_mapa_ocup)
df['ocupacion_encuestado'] = df['ocupacion_encuestado'].map({**_mapa_ocup, 7:'Sin trabajo remunerado'})

print(f"Renombradas: {len(nombres_cortos)} | Columnas totales: {df.shape[1]}")

## 7. Recodificaciones

In [ ]:
# ── Recoding: Habilidades Digitales Q8 (0='No', 1='Sí') ──────────────

q8_vars = [
    'Q8_10', 'Q8_11', 'Q8_12', 'Q8_13', 'Q8_15', 'Q8_16', 'Q8_1', 'Q8_2',
    'Q8_3', 'Q8_4', 'Q8_5', 'Q8_6', 'Q8_14', 'Q8_17', 'Q8_18', 'Q8_7',
    'Q8_8', 'Q8_9'
]

for col in q8_vars:
    if col in df.columns:
        df[col] = df[col].replace({0: 'No', 1: 'Sí'})

print(f'✓ Recoding Q8: 0→No, 1→Sí | {len(q8_vars)} variables procesadas')
print(f'  Ejemplo {q8_vars[0]}: {df[q8_vars[0]].value_counts().to_dict()}')

In [ ]:
df = df.copy()

# Identificación
df['region'] = df['region'].map({
    1:'Tarapacá', 2:'Antofagasta', 3:'Atacama', 4:'Coquimbo', 5:'Valparaíso',
    6:"O'Higgins", 7:'Maule', 8:'Biobío', 9:'Araucanía', 10:'Los Lagos',
    11:'Aysén', 12:'Magallanes', 13:'Metropolitana', 14:'Los Ríos', 15:'Arica y Parinacota', 16:'Ñuble'
})
df['zona'] = df['zona'].map({1:'Urbana', 2:'Rural'})

# Sociodemográficas del entrevistado
df['sexo'] = df['sexo'].map({1:'Hombre', 2:'Mujer'})
df['educ'] = df['educ'].map(_mapa_educ)
df['educ_grupo'] = df['educ'].map({
    'Sin educación formal':'Básica o menos', 'Básica incompleta':'Básica o menos',
    'Básica completa':'Básica o menos', 'Media CH incompleta':'Media',
    'Media TP incompleta':'Media', 'Media CH completa':'Media', 'Media TP completa':'Media',
    'Superior técnica incompleta':'Superior', 'Superior técnica completa':'Superior',
    'Superior universitaria incompleta':'Superior', 'Superior universitaria completa':'Superior',
})
df['tramo_edad'] = pd.cut(df['edad'], bins=[0,17,29,44,59,200],
                          labels=['Menor de 18','18-29','30-44','45-59','60 y más'], right=True)
df['actividad'] = df['actividad'].map({
    1:'Trabajador independiente', 2:'Empleador/patrón', 3:'Empleado dependiente',
    4:'Familiar no remunerado', 5:'FFAA y de orden', 6:'Cesante',
    7:'Jubilado/pensionado', 8:'Estudiante', 9:'Labores del hogar'
})

# Acceso a internet
df['acceso_internet_hogar'] = df['acceso_internet_hogar'].map({1:'Sí', 2:'No'})
df['tipo_acceso_fijo'] = df['tipo_acceso_fijo'].map({
    1:'ADSL', 2:'Cable/Módem', 3:'Fibra óptica', 4:'Inalámbrica',
    5:'Satelital', 31:'WiFi', 32:'Antena', 33:'Banda ancha', 34:'Acceso telefónico', 88:'No sabe'
})
df['velocidad_contratada'] = df['velocidad_contratada'].map({
    1:'Hasta 10 Mbps', 2:'Más de 10 a 100 Mbps', 3:'Más de 100 a 500 Mbps',
    4:'Más de 500 Mbps a 1 Gbps', 5:'Más de 1 Gbps', 99:'NS/NR'
})
df['tipo_plan'] = df['tipo_plan'].map({
    1:'Banda ancha desnuda', 2:'BA + TV Cable', 3:'BA + Telefonía fija',
    4:'Triple pack (BA+TV+Tel)', 5:'Otros planes'
})
df['tipo_acceso_mas_usado'] = df['tipo_acceso_mas_usado'].map({
    1.0:'Banda Ancha Fija / WiFi', 2.0:'Banda Ancha Móvil',
    3.0:'Internet Móvil (Smartphone/Tablet)', 4.0:'Conexión Satelital'
})

# Uso individual
df['uso_computador']  = df['uso_computador'].map({1:'Sí', 2:'No'})
df['uso_smartphone']  = df['uso_smartphone'].map({1:'Sí', 2:'No'})
df['ultimo_uso_internet'] = df['ultimo_uso_internet'].map({
    1:'Hoy', 2:'Entre 2 y 3 días', 3:'Entre 3 y 7 días', 4:'Entre 1 y 4 semanas',
    5:'Más de 4 semanas', 6:'Más de 12 meses', 7:'Nunca'
})
df['frecuencia_internet'] = df['frecuencia_internet'].map({
    1:'Todos los días', 2:'Varias veces por semana',
    3:'Al menos una vez al mes', 4:'Menos de una vez al mes'
})
df['tiempo_diario_internet'] = df['tiempo_diario_internet'].map({
    1:'Menos de 1 hora', 2:'Entre 1 y 2 horas', 3:'Entre 2 y 4 horas', 4:'Más de 4 horas'
})

# Percepciones
df['percepcion_proteccion']     = df['percepcion_proteccion'].map({
    1:'Muy protegido', 2:'Protegido', 3:'Desprotegido', 4:'Muy desprotegido', 99:'NS/NR'
})
df['internet_mejora_vida']      = df['internet_mejora_vida'].map({1:'Sí', 2:'No'})
df['internet_facilita_trabajo'] = df['internet_facilita_trabajo'].map({1:'Sí', 2:'No'})

print('Recodificaciones completadas.')
print(f"sexo: {df['sexo'].value_counts().to_dict()}")
print(f"acceso: {df['acceso_internet_hogar'].value_counts().to_dict()}")

## 8. Ingreso del hogar

In [ ]:
_rangos = {
    11:(0,129000),12:(130000,226000),13:(227000,393000),14:(394000,686000),15:(687000,1100000),16:(1200000,2000000),17:(2100000,None),
    21:(0,210000),22:(211000,366000),23:(367000,639000),24:(640000,1100000),25:(1200000,1900000),26:(2000000,3300000),27:(3400000,None),
    31:(0,279000),32:(280000,487000),33:(488000,849000),34:(850000,1400000),35:(1500000,2500000),36:(2600000,4500000),37:(4600000,None),
    41:(0,341000),42:(342000,595000),43:(596000,1000000),44:(1100000,1800000),45:(1900000,3100000),46:(3200000,5500000),47:(5600000,None),
    51:(0,399000),52:(400000,696000),53:(697000,1200000),54:(1300000,2100000),55:(2200000,3600000),56:(3700000,6400000),57:(6500000,None),
    61:(0,453000),62:(454000,791000),63:(792000,1300000),64:(1400000,2400000),65:(2500000,4100000),66:(4200000,7300000),67:(7400000,None),
    71:(0,505000),72:(506000,881000),73:(882000,1500000),74:(1600000,2600000),75:(2700000,4600000),76:(4700000,8100000),77:(8200000,None),
    81:(0,555000),82:(556000,967000),83:(968000,1600000),84:(1700000,2900000),85:(3000000,5100000),86:(5200000,8900000),87:(9000000,None),
    91:(0,602000),92:(603000,1000000),93:(1100000,1800000),94:(1900000,3100000),95:(3200000,5500000),96:(5600000,9700000),97:(9800000,None),
    101:(0,648000),102:(649000,1100000),103:(1200000,1900000),104:(2000000,3400000),105:(3500000,5900000),106:(6000000,10400000),107:(10500000,None),
}
_mapa_pm = {float(k): (v[0]*1.5 if v[1] is None else (v[0]+v[1])/2) for k, v in _rangos.items()}

df['ingreso_pm'] = df['ingreso_hogar'].map(_mapa_pm)
df['ingreso_tramo'] = pd.cut(
    df['ingreso_pm'],
    bins=[0, 384000, 540000, 798000, 1100000, 1700000, float('inf')],
    labels=['Hasta $384 mil','$384 mil a $540 mil','$540 mil a $798 mil',
            '$798 mil a $1,1 millón','$1,1 millón a $1,7 millones','Más de $1,7 millones'],
    right=True
)
df['ingreso_grupo'] = df['ingreso_tramo'].map({
    'Hasta $384 mil':'Bajo', '$384 mil a $540 mil':'Bajo',
    '$540 mil a $798 mil':'Medio', '$798 mil a $1,1 millón':'Medio',
    '$1,1 millón a $1,7 millones':'Alto', 'Más de $1,7 millones':'Alto',
})

# Validación: promedio de ingreso debe subir de E a AB
(
    df.groupby('gse', observed=True)['ingreso_pm']
    .agg(N='count', Promedio='mean').reindex(_ORDEN_GSE).round(0).astype({'N':int,'Promedio':int})
)

## 9. Funciones de análisis ponderado

In [ ]:
ORDEN_CATEGORIAS = {
    'sexo':         ['Hombre','Mujer'],
    'zona':         ['Urbana','Rural'],
    'region':       ['Tarapacá','Antofagasta','Atacama','Coquimbo','Valparaíso',"O'Higgins",'Maule',
                     'Biobío','Araucanía','Los Lagos','Aysén','Magallanes','Metropolitana',
                     'Los Ríos','Arica y Parinacota','Ñuble'],
    'educ':         ['Sin educación formal','Básica incompleta','Básica completa',
                     'Media CH incompleta','Media TP incompleta','Media CH completa','Media TP completa',
                     'Superior técnica incompleta','Superior técnica completa',
                     'Superior universitaria incompleta','Superior universitaria completa'],
    'educ_grupo':   ['Básica o menos','Media','Superior'],
    'tramo_edad':   ['Menor de 18','18-29','30-44','45-59','60 y más'],
    'actividad':    ['Trabajador independiente','Empleador/patrón','Empleado dependiente',
                     'Familiar no remunerado','FFAA y de orden','Cesante',
                     'Jubilado/pensionado','Estudiante','Labores del hogar'],
    'ocupacion_jh': ['Trabajos ocasionales e informales','Oficio menor - obrero no calificado',
                     'Obrero calificado - microempresario','Empleado medio - técnico - prof. independiente',
                     'Ejecutivo medio - prof. universitario','Alto ejecutivo - empresario - directivo'],
    'ocupacion_encuestado': ['Trabajos ocasionales e informales','Oficio menor - obrero no calificado',
                     'Obrero calificado - microempresario','Empleado medio - técnico - prof. independiente',
                     'Ejecutivo medio - prof. universitario','Alto ejecutivo - empresario - directivo',
                     'Sin trabajo remunerado'],
    'gse':              ['AB', 'C1', 'C2', 'C3', 'D', 'E'],
    'ingreso_tramo':    ['Hasta $384 mil','$384 mil a $540 mil','$540 mil a $798 mil',
                         '$798 mil a $1,1 millón','$1,1 millón a $1,7 millones','Más de $1,7 millones'],
    'ingreso_grupo':    ['Bajo','Medio','Alto'],
    'acceso_internet_hogar':    ['Sí','No'],
    'uso_computador':           ['Sí','No'],
    'uso_smartphone':           ['Sí','No'],
    'internet_mejora_vida':     ['Sí','No'],
    'internet_facilita_trabajo':['Sí','No'],
    'tipo_acceso_fijo':         ['Fibra óptica','Cable/Módem','ADSL','Inalámbrica','Satelital','WiFi','Antena','Banda ancha','Acceso telefónico','No sabe'],
    'tipo_acceso_mas_usado':    ['Banda Ancha Fija / WiFi','Banda Ancha Móvil','Internet Móvil (Smartphone/Tablet)','Conexión Satelital'],
    'tipo_plan':                ['Banda ancha desnuda','BA + TV Cable','BA + Telefonía fija','Triple pack (BA+TV+Tel)','Otros planes'],
    'ultimo_uso_internet':      ['Hoy','Entre 2 y 3 días','Entre 3 y 7 días',
                                  'Entre 1 y 4 semanas','Más de 4 semanas','Más de 12 meses','Nunca'],
    'frecuencia_internet':      ['Todos los días','Varias veces por semana',
                                  'Al menos una vez al mes','Menos de una vez al mes'],
    'tiempo_diario_internet':   ['Menos de 1 hora','Entre 1 y 2 horas','Entre 2 y 4 horas','Más de 4 horas'],
    'percepcion_proteccion':    ['Muy protegido','Protegido','Desprotegido','Muy desprotegido','NS/NR'],
    'velocidad_contratada':     ['Hasta 10 Mbps','Más de 10 a 100 Mbps','Más de 100 a 500 Mbps',
                                  'Más de 500 Mbps a 1 Gbps','Más de 1 Gbps','NS/NR'],
}


def fordf(df_tabla, titulo=None):
    """Formato visual: porcentajes con 1 decimal."""
    
    # 1. Identificar solo las columnas que son numéricas
    num_cols = df_tabla.select_dtypes(include=['number']).columns
    
    # 2. Aplicar el formato solo a esas columnas
    estilo = df_tabla.style.format({
        col: '{:.1f}'
        for col in num_cols
    })
    
    if titulo: 
        estilo = estilo.set_caption(titulo)
        
    return estilo




def _ordenar(df_res, var, cruzada=False):
    if var not in ORDEN_CATEGORIAS: return df_res
    orden = ORDEN_CATEGORIAS[var]
    if cruzada:
        ok  = [v for v in orden if v in df_res.index]
        rst = [v for v in df_res.index if v not in ok and v != 'Total']
        fin = ok + rst + (['Total'] if 'Total' in df_res.index else [])
        return df_res.reindex(fin)
    ok  = [v for v in orden if v in df_res[var].values]
    rst = [v for v in df_res[var].values if v not in ok and v != 'Total']
    df_res[var] = pd.Categorical(df_res[var], categories=ok+rst+['Total'], ordered=True)
    return df_res.sort_values(var).reset_index(drop=True)


def dstats(data_df, variables, tipo='frecuencia', cruce=None, factor=None, transponer=False, estilo=True):
    """
    Análisis ponderado de variables simples — solo porcentajes.
    tipo: 'frecuencia' | 'cruzada' | 'promedio' | 'suma'
    Si estilo=True, retorna Styler formateado. Si es False, retorna el DataFrame puro.
    Ejemplo: dstats(df, 'sexo', tipo='frecuencia', factor='fe_personas', estilo=False)
    """
    if isinstance(variables, str): variables = [variables]
    for col in variables + [factor] + ([cruce] if cruce else []):
        if col not in data_df.columns:
            raise ValueError(f"Columna '{col}' no existe.")

    if tipo == 'frecuencia':
        var = variables[0]
        tot = data_df[factor].sum()
        res = data_df.groupby(var, observed=True)[factor].sum().reset_index().rename(columns={factor:'porcentaje'})
        res['porcentaje'] = (res['porcentaje'] / tot * 100).round(2)
        res = _ordenar(res, var).set_index(var)
        
        if estilo:
            titulo = f"Frecuencia: '{var}' — base ponderada: {tot:,.0f} ({factor})"
            return fordf(res, titulo=titulo)
        return res

    if tipo == 'cruzada':
        var = variables[0]
        tot = data_df[factor].sum()
        t   = data_df.pivot_table(values=factor, index=var, columns=cruce, aggfunc='sum', fill_value=0, observed=False)
        tp  = t.div(t.sum(axis=0), axis=1).mul(100).round(2)
        if var in ORDEN_CATEGORIAS:
            of = [v for v in ORDEN_CATEGORIAS[var] if v in t.index];  tp = tp.reindex(of)
        if cruce in ORDEN_CATEGORIAS:
            oc = [v for v in ORDEN_CATEGORIAS[cruce] if v in tp.columns]; tp = tp[oc]
        if transponer: tp = tp.T
        res = tp
        
        if estilo:
            titulo = f"Cruce: '{var}' según '{cruce}' — base ponderada: {tot:,.0f} ({factor})"
            return fordf(res, titulo=titulo)
        return res

    def _wavg(sub, v, f):
        d = sub[[v, f]].dropna()
        return float(round(np.average(d[v], weights=d[f]), 4)) if len(d) > 0 and d[f].sum() > 0 else np.nan

    def _wsum(sub, v, f):
        d = sub[[v, f]].dropna()
        return float(round((d[v]*d[f]).sum(), 4))

    fn = _wavg if tipo == 'promedio' else _wsum
    col_name = 'promedio_ponderado' if tipo == 'promedio' else 'suma_ponderada'

    if not cruce:
        res = pd.DataFrame([(v, fn(data_df, v, factor)) for v in variables], columns=['variable', col_name])
        
        if estilo:
            titulo = f"{tipo.capitalize()} de variables — factor: {factor}"
            return fordf(res, titulo=titulo)
        return res

    filas = {g: {v: fn(sg, v, factor) for v in variables} for g, sg in data_df.groupby(cruce, observed=True)}
    res = pd.DataFrame(filas).T
    res.index.name = cruce
    if cruce in ORDEN_CATEGORIAS:
        ok  = [v for v in ORDEN_CATEGORIAS[cruce] if v in res.index]
        rst = [v for v in res.index if v not in ok and v != 'Total']
        res = res.reindex(ok + rst + ['Total'])
        
    if estilo:
        titulo = f"{tipo.capitalize()} cruzado por '{cruce}' — factor: {factor}"
        return fordf(res, titulo=titulo)
    return res


print('ORDEN_CATEGORIAS, fordf, dstats listos (solo porcentajes).')

## 10. Grupos de respuesta múltiple

In [ ]:
_c = df.columns

GRUPOS_RM = {
    # Hogar
    'A12':  ('Pueblos indígenas o tribales (hogar)',            [c for c in _c if c.startswith('A12_') and not c.startswith('A12_1')]),
    'A13':  ('Condiciones permanentes de salud en el hogar',   [c for c in _c if c.startswith('A13_')]),
    'A14':  ('Situaciones laborales en el hogar',              [c for c in _c if c.startswith('A14_') and not c.endswith('_OTRA')]),
    # Acceso y conectividad
    'P3':   ('Dispositivos usados para acceder a internet',    [c for c in _c if c.startswith('P3_') and not c.endswith('_OTRA')]),
    'P4':   ('Formas de acceso pagado a internet en el hogar', [c for c in _c if c.startswith('P4_')]),
    'P6':   ('Razones para tener internet fijo',               [c for c in _c if c.startswith('P6_') and not c.startswith('P6_1_') and not c.endswith('_OTRA')]),
    'P6_1': ('Razones para tener internet móvil',              [c for c in _c if c.startswith('P6_1_')]),
    'P7':   ('Medidas de protección internet para menores',    [c for c in _c if c.startswith('P7_')]),
    'P8_3': ('Violencia digital en niños y adolescentes',      [c for c in _c if c.startswith('P8_3_')]),
    'P9':   ('Dispositivos de uso personal de menores',        [c for c in _c if c.startswith('P9_')]),
    'P12':  ('Conexión móvil 4G/5G',                           ['P12_11','P12_21','P12_31','P12_41']),
    'P13':  ('Razones de no acceso a internet fijo',           [c for c in _c if c.startswith('P13_') and not c.endswith('_OTRA')]),
    'P16':  ('Equipos que le interesaría tener (sin internet)',[c for c in _c if c.startswith('P16_')]),
    # Uso individual
    'Q6':   ('¿Cómo aprendió a usar el computador?',           [c for c in _c if c.startswith('Q6_') and c not in ['Q6_1','Q6_OTRA']]),
    'Q7_2': ('Smartphone 4G/5G',                               ['Q7_2_1','Q7_2_2','Q7_2_3','Q7_2_4']),
    'Q8':   ('Habilidades digitales',                          [c for c in _c if c.startswith('Q8_')]),
    'Q11_1':('Lugares donde usó internet ayer',                [c for c in _c if c.startswith('Q11_1_')]),
    'Q12':  ('Tipos de acceso en últimos 3 meses',             [c for c in _c if c.startswith('Q12_')]),
    'Q20':  ('Lugares donde usó internet fuera del hogar',     [c for c in _c if c.startswith('Q20_')]),
    'Q21':  ('Actividades realizadas en internet',             [c for c in _c if c.startswith('Q21_') and c not in ['Q21_1','Q21_10','Q21_19','Q21_26','Q21_33','Q21_38','Q21_44']]),
    'Q28':  ('Bienes y servicios comprados en internet',       [c for c in _c if c.startswith('Q28_') and not c.endswith('_OTRA')]),
    'Q32':  ('Actividades de seguridad y privacidad',          [c for c in _c if c.startswith('Q32_') and not c.endswith('_OTRA')]),
    'Q33':  ('Problemas de seguridad sufridos',                [c for c in _c if c.startswith('Q33_') and not c.endswith('_OTRA')]),
    'Q34':  ('Razones de no uso de internet',                  [c for c in _c if c.startswith('Q34_') and not c.endswith('_OTRA')]),
    'Q37':  ('Actividades de internet realizadas por terceros',[c for c in _c if c.startswith('Q37_')]),
    'Q39':  ('Equipos que le interesaría tener (no usuarios)', [c for c in _c if c.startswith('Q39_')]),
}

def analizar_rm(grupo, factor='fe_hogar', top_n=None, estilo=True):
    """
    Analiza un grupo de respuesta múltiple — solo porcentajes.
    Si estilo=True, retorna tabla estilizada (HTML). Si es False, retorna el DataFrame puro.
    """
    if grupo not in GRUPOS_RM: 
        raise ValueError(f"Grupo '{grupo}' no existe.")
        
    desc, cols = GRUPOS_RM[grupo]
    cols = [c for c in cols if c in df.columns]
    base = df.loc[df[cols].notna().any(axis=1), factor].sum()
    
    filas = [
        {'variable': c,
         'etiqueta': etiquetas_limpias.get(c, c),
         'porcentaje': round(df.loc[df[c]==1, factor].sum() / base * 100, 1)}
        for c in cols
    ]
    res = pd.DataFrame(filas).sort_values('porcentaje', ascending=False).reset_index(drop=True)
    if top_n: res = res.head(top_n)
    res.index += 1
    
    if estilo:
        titulo_tabla = f"{desc} — base ponderada: {int(base):,} ({factor})"
        return fordf(res, titulo=titulo_tabla)
    return res

print("Grupos de respuesta múltiple disponibles:")
for k, (desc, cols) in GRUPOS_RM.items():
    cols_validas = [c for c in cols if c in df.columns]
    print(f"  '{k}': {desc} ({len(cols_validas)} opciones)")

In [ ]:
def analizar_rm_cruce(grupo, cruce, factor='fe_personas', estilo=True):
    """
    Analiza un grupo de respuesta múltiple cruzado por una variable demográfica — solo porcentajes.
    Si estilo=True, retorna tabla estilizada (HTML). Si es False, retorna el DataFrame puro.
    """
    if grupo not in GRUPOS_RM: 
        raise ValueError(f"Grupo '{grupo}' no existe.")
        
    desc, cols = GRUPOS_RM[grupo]
    cols = [c for c in cols if c in df.columns]
    
    base_cruce = df.loc[df[cols].notna().any(axis=1)].groupby(cruce, observed=True)[factor].sum()
    
    resultados = {}
    for categoria in base_cruce.index:
        base = base_cruce[categoria]
        df_cat = df[df[cruce] == categoria]
        
        pcts = {
            etiquetas_limpias.get(c, c): round((df_cat.loc[df_cat[c]==1, factor].sum() / base) * 100, 1) 
            if base > 0 else 0 
            for c in cols
        }
        resultados[categoria] = pcts
        
    res_df = pd.DataFrame(resultados)
    res_df = res_df.sort_values(by=res_df.columns[0], ascending=False)
    
    if estilo:
        titulo_tabla = f"{desc} cruzado por '{cruce}' — factor: {factor}"
        return fordf(res_df, titulo=titulo_tabla)
    return res_df

## 11. Habilidades digitales Q8

In [ ]:
# ═ DEBUG: Inspeccionar valores de Q8 ═
print("Checkeando tipos y valores de columnas Q8...")
q8_cols = [c for c in df.columns if c.startswith('Q8')]
print(f"Columnas Q8: {len(q8_cols)}")

for col in q8_cols[:3]:
    print(f"\n{col}:")
    print(f"  dtype: {df[col].dtype}")
    print(f"  primeros 5: {df[col].head(5).tolist()}")
    print(f"  únicos (primeros 5): {df[col].unique()[:5].tolist()}")
    print(f"  tipo primer valor: {type(df[col].iloc[0])}")

### Habilidades por nivel

In [ ]:
# Clasificación de habilidades Q8 por nivel de dificultad (PONDERADO)
# Valores: 'Sí', 'No', NaN

Q8_BASICO = {
    'Q8_10': 'Streaming (video/música)',
    'Q8_11': 'Juegos en línea',
    'Q8_12': 'Revisar redes sociales',
    'Q8_13': 'Publicar en redes sociales',
    'Q8_15': 'Videollamadas',
    'Q8_16': 'Correo electrónico',
}
Q8_INTERMEDIO = {
    'Q8_1':  'Procesador de texto (Word)',
    'Q8_2':  'Planilla de cálculo (Excel)',
    'Q8_3':  'Presentaciones (PowerPoint)',
    'Q8_4':  'Transferir archivos / nube',
    'Q8_5':  'Conectar nuevo dispositivo',
    'Q8_6':  'Instalar y configurar apps',
    'Q8_14': 'Editar fotos o videos',
    'Q8_17': 'Transacciones y pagos en línea',
    'Q8_18': 'Uso de IA (ChatGPT, etc.)',
}
Q8_AVANZADO = {
    'Q8_7': 'Configurar seguridad del dispositivo',
    'Q8_8': 'Instalar SO / programar (Python, Java…)',
    'Q8_9': 'Crear un sitio web',
}

_cols_b  = list(Q8_BASICO)
_cols_i  = list(Q8_INTERMEDIO)
_cols_a  = list(Q8_AVANZADO)
_cols_q8 = _cols_b + _cols_i + _cols_a + ['Q8_19']

def _nivel(row):
    """Clasifica nivel — compara con 'Sí' (no con 1)."""
    if any(row[c] == 'Sí' for c in _cols_a if pd.notna(row[c])):
        return 'Avanzado'
    if any(row[c] == 'Sí' for c in _cols_i if pd.notna(row[c])):
        return 'Intermedio'
    if any(row[c] == 'Sí' for c in _cols_b if pd.notna(row[c])):
        return 'Básico'
    return 'Sin habilidades'

# Crear nivel_habilidades
df['nivel_habilidades'] = df.apply(_nivel, axis=1)
mask_q8 = df[_cols_q8].notna().any(axis=1)

# Análisis ponderado por fe_personas
factor = 'fe_personas'
base_ponderada = df.loc[mask_q8, factor].sum()
base_ponderada_int = int(base_ponderada)

print(f"Base ponderada (fe_personas): {base_ponderada_int:,}\n")
print("Distribución por nivel (PONDERADA):")

orden_nivel = ['Avanzado', 'Intermedio', 'Básico', 'Sin habilidades']
for niv in orden_nivel:
    n_pond = df.loc[mask_q8 & (df['nivel_habilidades'] == niv), factor].sum()
    pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
    print(f"  {niv:<20} {int(n_pond):>10,}  ({pct:.1f}%)")

print("\n── Habilidades por nivel (PONDERADO) ──────────────────────────────────────────────")
for nivel, items in [('BÁSICO', Q8_BASICO), ('INTERMEDIO', Q8_INTERMEDIO), ('AVANZADO', Q8_AVANZADO)]:
    print(f"\n{nivel}")
    for cod, desc in items.items():
        n_pond = df.loc[mask_q8 & (df[cod] == 'Sí'), factor].sum()
        pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
        print(f"  {cod:<8} {desc:<45}  {int(n_pond):>10,} ({pct:.1f}%)")

### Habilidades por tipo de uso

In [ ]:
# ── Clasificación de Habilidades Digitales Q8 por tipo de uso (PONDERADO) ──────────

Q8_PRODUCTIVAS = {
    'Q8_1':  'Procesador de texto (Word)',
    'Q8_2':  'Planilla de cálculo (Excel)',
    'Q8_3':  'Presentaciones (PowerPoint)',
    'Q8_4':  'Transferir archivos / nube',
    'Q8_5':  'Conectar nuevo dispositivo',
    'Q8_6':  'Instalar y configurar apps',
    'Q8_7':  'Configurar seguridad del dispositivo',
    'Q8_8':  'Instalar SO / programar (Python, Java…)',
    'Q8_17': 'Transacciones y pagos en línea',
    'Q8_18': 'Uso de IA (ChatGPT, etc.)',
}

Q8_RECREATIVAS = {
    'Q8_10': 'Streaming (video/música)',
    'Q8_11': 'Juegos en línea',
    'Q8_14': 'Editar fotos o videos',
    'Q8_9':  'Crear un sitio web',
}

Q8_COMUNICACIONES = {
    'Q8_12': 'Revisar redes sociales',
    'Q8_13': 'Publicar en redes sociales',
    'Q8_15': 'Videollamadas',
    'Q8_16': 'Correo electrónico',
}

_cols_prod  = list(Q8_PRODUCTIVAS)
_cols_rec   = list(Q8_RECREATIVAS)
_cols_com   = list(Q8_COMUNICACIONES)
_cols_q8_tipos = _cols_prod + _cols_rec + _cols_com + ['Q8_19']

def _tipo_principal(row):
    """Clasifica tipo principal de habilidades — primero verifica productivas, luego recreativas, luego comunicaciones."""
    if any(row[c] == 'Sí' for c in _cols_prod if pd.notna(row[c])):
        return 'Productivas'
    if any(row[c] == 'Sí' for c in _cols_rec if pd.notna(row[c])):
        return 'Recreativas'
    if any(row[c] == 'Sí' for c in _cols_com if pd.notna(row[c])):
        return 'Comunicaciones'
    return 'Sin habilidades'

# Crear tipo_habilidades_principal
df['tipo_habilidades_principal'] = df.apply(_tipo_principal, axis=1)
mask_q8 = df[_cols_q8_tipos].notna().any(axis=1)

# Análisis ponderado por fe_personas
factor = 'fe_personas'
base_ponderada = df.loc[mask_q8, factor].sum()
base_ponderada_int = int(base_ponderada)

print(f"Base ponderada (fe_personas): {base_ponderada_int:,}\n")
print("Distribución por tipo (PONDERADA):")

orden_tipo = ['Productivas', 'Recreativas', 'Comunicaciones', 'Sin habilidades']
for tipo in orden_tipo:
    n_pond = df.loc[mask_q8 & (df['tipo_habilidades_principal'] == tipo), factor].sum()
    pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
    print(f"  {tipo:<20} {int(n_pond):>10,}  ({pct:.1f}%)")

print("\n── Habilidades por tipo (PONDERADO) ──────────────────────────────────────────────")
for tipo, items in [('PRODUCTIVAS', Q8_PRODUCTIVAS), ('RECREATIVAS', Q8_RECREATIVAS), ('COMUNICACIONES', Q8_COMUNICACIONES)]:
    print(f"\n{tipo}")
    for cod, desc in items.items():
        n_pond = df.loc[mask_q8 & (df[cod] == 'Sí'), factor].sum()
        pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
        print(f"  {cod:<8} {desc:<45}  {int(n_pond):>10,} ({pct:.1f}%)")

# Análisis

## Caracterización Muestral Personas

In [ ]:
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'region', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'zona', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'educ', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'ocupacion_encuestado', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'tramo_edad', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'gse', tipo='frecuencia', factor='fe_personas'))


## Caracterización muestral hogares

In [83]:
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'region', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'zona', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'educ', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'ocupacion_encuestado', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'tramo_edad', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'gse', tipo='frecuencia', factor='fe_hogar'))


,porcentaje
sexo,
Hombre,45.2
Mujer,54.8


,porcentaje
region,
Tarapacá,1.7
Antofagasta,3.1
Atacama,1.6
Coquimbo,4.2
Valparaíso,10.8
O'Higgins,5.3
Maule,6.2
Biobío,9.0
Araucanía,5.6


,porcentaje
zona,
Urbana,87.6
Rural,12.4


,porcentaje
educ,
Sin educación formal,0.5
Básica incompleta,5.9
Básica completa,7.8
Media CH incompleta,7.3
Media TP incompleta,3.3
Media CH completa,23.8
Media TP completa,9.2
Superior técnica incompleta,2.9
Superior técnica completa,11.2


,porcentaje
ocupacion_encuestado,
Trabajos ocasionales e informales,10.1
Oficio menor - obrero no calificado,7.9
Obrero calificado - microempresario,25.2
Empleado medio - técnico - prof. independiente,30.1
Ejecutivo medio - prof. universitario,8.8
Alto ejecutivo - empresario - directivo,1.0
Sin trabajo remunerado,16.9


,porcentaje
sexo,
Hombre,45.2
Mujer,54.8


,porcentaje
tramo_edad,
Menor de 18,0.8
18-29,17.8
30-44,26.4
45-59,23.3
60 y más,31.8


,porcentaje
gse,
AB,10.0
C1,12.6
C2,22.4
C3,24.0
D,15.8
E,15.2


## Conectividad e infra

### Nacional (sin cruces)

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(analizar_rm('P3', factor='fe_hogar', estilo=True))
display(analizar_rm('P4', factor='fe_hogar', estilo=True))
display(analizar_rm('P6', factor='fe_hogar', estilo=True))
display(analizar_rm('P6_1', factor='fe_hogar', estilo=True))
display(analizar_rm('P7', factor='fe_hogar', estilo=True))
display(analizar_rm('P9', factor='fe_hogar', estilo=True))
display(analizar_rm('P8_3', factor='fe_hogar', estilo=True))


### POR GSE

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='gse', factor='fe_hogar', estilo=True))

### POR REGION

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='region', factor='fe_hogar', estilo=True))

### POR ZONA

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='zona', factor='fe_hogar', estilo=True))

### POR EDUCACIÓN JEFE DE HOGAR

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='educ_jh', factor='fe_hogar', estilo=True))

### POR OCUPACIÓN JEFE DE HOGAR

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))

### POR SEXO

In [ ]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='sexo', factor='fe_hogar', estilo=True))